In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze = spark.table("nyc_taxi.bronze.yellow_tripdata")

DISTANCE_CEILING = 100
TOLLS_CEILING    = 100

# trip_id computed EARLY now, before tagging, so duplicate detection can use it as a rule condition
key_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount"]
bronze = bronze.withColumn("trip_id",
    F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in key_cols]), 256))

# Stable tiebreaker so ranking is deterministic even when rows are 100% identical
bronze = bronze.withColumn("_row_uid", F.monotonically_increasing_id())
dupe_window = Window.partitionBy("trip_id").orderBy("_row_uid")
bronze = bronze.withColumn("_dupe_rank", F.row_number().over(dupe_window))

bronze = bronze.withColumn("_payment_type_safe", F.coalesce(F.col("payment_type"), F.lit(-1)))

tagged = bronze.withColumn("_reject_reason",
    F.when(F.col("tpep_pickup_datetime").isNull() | F.col("tpep_dropoff_datetime").isNull(),
           "missing_timestamp")
     .when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"),
           "dropoff_before_pickup")
     .when(F.col("_dupe_rank") > 1,
           "duplicate_source_row")
     .when(F.year(F.col("tpep_pickup_datetime")) != 2024,
           "implausible_pickup_year")
     .when(F.col("trip_distance") <= 0,
           "zero_or_negative_distance")
     .when(F.col("trip_distance") > DISTANCE_CEILING,
           "implausible_distance")
     .when(F.col("passenger_count").isNull(),
           "null_passenger_count")
     .when(F.col("passenger_count") == 0,
           "zero_passenger_count")
     .when(F.col("passenger_count") > 6,
           "implausible_passenger_count")
     .when(F.col("tolls_amount") > TOLLS_CEILING,
           "implausible_tolls")
     .when(F.col("fare_amount") < 0,
           "negative_fare")
     .when((F.col("fare_amount") < 3.00) & (~F.col("_payment_type_safe").isin(3, 4)),
           "fare_below_minimum")
     .when(F.col("total_amount") < 0,
           "negative_total")
     .when((F.col("total_amount") < 4.50) & (~F.col("_payment_type_safe").isin(3, 4)),
           "total_below_minimum")
     .otherwise(None)
).drop("_payment_type_safe", "_row_uid", "_dupe_rank")

clean      = tagged.filter(F.col("_reject_reason").isNull()).drop("_reject_reason")
quarantine = tagged.filter(F.col("_reject_reason").isNotNull())

clean = (clean
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("_silver_processed_at", F.current_timestamp())
)

print("clean:     ", clean.count())
print("quarantine:", quarantine.count())
quarantine.groupBy("_reject_reason").count().orderBy(F.desc("count")).show()